In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys, os

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
pd.set_option('display.max_columns', None)

In [34]:
from routes.Paths import(
    Ruta_Viajes,
    Ruta_global,
    Ruta_local
)
from utils.functions import load_files_from_folder
from utils.Data_ModeloComercial.b_processing_crm_ventas import process_crm_ventas

In [36]:
required_columns =[
"fecha",
"hora",
"agencia",
"estado",
"servicios",
"tipo",
"total dinero",
"numero de documento del titular"
]

df_viajes = load_files_from_folder(
    Ruta_Viajes,
    set(required_columns),
    required_columns
)

df_viajes["fecha_transaccion"] = pd.to_datetime(
                                    df_viajes["fecha"].astype(str) + " " + df_viajes["hora"].astype(str)
)
columns_viajes =[
"fecha_transaccion",
"fecha",
"agencia",
"estado",
"servicios",
"tipo",
"total dinero",
"numero de documento del titular",
"agencia",
"source_file",
"file_date"
]

df_viajes = df_viajes[columns_viajes]

df_viajes = df_viajes.rename(columns={
    "fecha":"date",
    "numero de documento del titular":"documento_cliente",
})

df_viajes["documento_cliente"] = df_viajes["documento_cliente"].astype(str)
df_viajes["date"] = pd.to_datetime(df_viajes["date"]).dt.date
df_viajes["servicios"] = pd.to_numeric(df_viajes["servicios"])

In [37]:
df_crm = process_crm_ventas(save_file = False)
columnas_crm_requeridas = [
    "NÚMERO_DOCUMENTO_DEL_CLIENTE",
    "RADICADO",
    "IdTipificacion",
    "NIVEL_1",
    "NIVEL_2",
    "NIVEL_3",
    "TIPO_DE_LLAMADA",
    "NOMBRE_CLIENTE",
    "CONTACTO_DE_CLIENTE",
    "NOMBRE_ALIADO",
    "Id_opl",
    "NOMBRE_ASESOR",
    "Correo_contratista",
    "FECHA_DE_INICIO",
    "START_DATE",
    "FECHA_FIN",
    "END_DATE",
    "START_MONTH_DATE",
    "OFRECIMIENTO_EFECTIVO",
    "Aplica_Ofrecimiento",
    "VALOR_PUNTOS",
    "CANAL_DE_ATENCIÓN",
    "Colas",
    "SKILL",
    "IdMasterSkill",
    "ESTADO_GESTIÓN",
    "NÚMERO_PEDIDO"
]

df_crm["FECHA_DE_INICIO"] = pd.to_datetime(
    df_crm["FECHA_DE_INICIO"],
    errors="coerce"
)

df_crm["FECHA_FIN"] = pd.to_datetime(
    df_crm["FECHA_FIN"],
    errors="coerce"
)

df_crm["Prioridad"] = (
    df_crm["OFRECIMIENTO_EFECTIVO"]
        .map({"Si": 1, "No": 2})
)

df_crm = df_crm.sort_values(
    by=[
        "NÚMERO_DOCUMENTO_DEL_CLIENTE",
        "START_DATE",
        "Prioridad"
    ]
)
df_crm["NÚMERO_DOCUMENTO_DEL_CLIENTE"] = df_crm["NÚMERO_DOCUMENTO_DEL_CLIENTE"].astype(str)

In [38]:
df_viajes = df_viajes.merge(
    df_crm[columnas_crm_requeridas],
        how="left",
        left_on=[
            "documento_cliente",
            "date"
        ],
        right_on=[
            "NÚMERO_DOCUMENTO_DEL_CLIENTE",
            "START_DATE"
        ]
    ).drop(columns=["NÚMERO_DOCUMENTO_DEL_CLIENTE","START_DATE"])

df_viajes["HORAS_DESDE_INICIO"] = ((df_viajes["fecha_transaccion"] - df_viajes["FECHA_DE_INICIO"]).dt.total_seconds()/3600).round(2)
df_viajes["HORAS_DESDE_FIN"] = ((df_viajes["fecha_transaccion"] - df_viajes["FECHA_FIN"]).dt.total_seconds()/3600).round(2)

df_viajes["VALIDACION_VENTA"] = np.where(
    (df_viajes["OFRECIMIENTO_EFECTIVO"].eq("Si")) & 
    (df_viajes["HORAS_DESDE_INICIO"].between(0, 24) | df_viajes["HORAS_DESDE_FIN"].between(0, 24)),
    "Venta efectiva",
    "Venta no efectiva"
)

In [40]:
df_viajes["VALIDACION_VENTA"].value_counts()

VALIDACION_VENTA
Venta no efectiva    30515
Venta efectiva         167
Name: count, dtype: int64